# ViT Baseline -- Phase 2: Multitask Fine-tuning (Classification + Mass & pT Regression)

**Baseline comparison for `xcit-multitask.ipynb`.**

Loads the ViT encoder pretrained in `vit-jet-images-mae.ipynb` and fine-tunes it
with identical protocol to the XCiT multitask notebook.

| | ViT (this notebook) | XCiT |
|---|---|---|
| Encoder | Standard MHSA | Cross-covariance XCA + LPI |
| Pretrained weights | `best_vit_mae.pth` | `best_xcit_mae.pth` |
| Fine-tune protocol | Identical (2-step) | Identical |
| Heads | Identical | Identical |
| Loss | Identical | Identical |

Any difference in results is purely attributable to the encoder architecture.

## 1 . Installs

In [12]:
# !pip install torch torchvision einops h5py scikit-learn matplotlib


## 2 . Imports & device

In [13]:
import numpy as np, math, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f'Device: {DEVICE}')


Device: mps


## 3 . Hyperparameters

In [14]:
# -- Paths ------------------------------------------------------------------
VIT_WEIGHTS = './models/baseline-vit/best_vit_mae.pth'
MT_DIR      = './data-concat'
BATCH_SIZE  = 16
NUM_WORKERS = 0

# -- Training (identical to xcit-multitask.ipynb) ----------------------------
EPOCHS_FROZEN   = 3
EPOCHS_FINETUNE = 1
EPOCHS_SCRATCH  = 3
LR_HEAD         = 1e-3
LR_BACKBONE_FT  = 3e-5
LR_SCRATCH      = 1e-3
WEIGHT_DECAY    = 1e-2
WARMUP_EPOCHS   = 5

W_CLS  = 2.0
W_MASS = 1.0
W_PT   = 1.0

# -- Architecture (must match vit-jet-images-mae.ipynb) ----------------------
IMAGE_SIZE  = 125
PATCH_SIZE  = 5
IN_CHANNELS = 8
NUM_CLASSES = 2
DIM         = 192
DEPTH       = 8
NUM_HEADS   = 8
MLP_RATIO   = 4
DROPOUT     = 0.2   # re-enabled for fine-tuning
DROP_PATH   = 0.05  # re-enabled for fine-tuning

print(f'ViT baseline  dim={DIM}  depth={DEPTH}  heads={NUM_HEADS}')
print(f'Fine-tune: freeze {EPOCHS_FROZEN}ep -> backbone LR={LR_BACKBONE_FT}  heads LR={LR_HEAD}')


ViT baseline  dim=192  depth=8  heads=8
Fine-tune: freeze 3ep -> backbone LR=3e-05  heads LR=0.001


## 4 . ViT Building Blocks (identical to Phase 1)
Must not be modified -- key names must match `ViTEncoder` in Phase 1 exactly.

In [15]:
class DropPath(nn.Module):
    def __init__(self, p=0.0):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0.0: return x
        keep = 1 - self.p
        mask = (torch.rand((x.shape[0],)+(1,)*(x.ndim-1), device=x.device)<keep).float()
        return x * mask / keep


class MHSA(nn.Module):
    # Standard Multi-Head Self-Attention. O(N^2 * d).
    def __init__(self, dim, num_heads=8, qkv_bias=True, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj      = nn.Sequential(nn.Linear(dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        B, N, C = x.shape
        h   = self.num_heads
        qkv = self.qkv(x).reshape(B, N, 3, h, C//h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


class ViTBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0, drop_path=0.0):
        super().__init__()
        self.norm1     = nn.LayerNorm(dim)
        self.norm2     = nn.LayerNorm(dim)
        self.attn      = MHSA(dim, num_heads=num_heads, dropout=dropout)
        self.ffn       = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim), nn.Dropout(dropout),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.ffn(self.norm2(x)))
        return x


print('ViT building blocks defined.')


ViT building blocks defined.


## 5 . ViT Multitask Model
Backbone key names exactly match `ViTEncoder` from Phase 1.
The three heads (`head_cls`, `head_mass`, `head_pt`) are always randomly initialised.

**Key structural difference from XCiT:** uses the CLS token directly from the last
ViT block output (no separate class-attention aggregation stage as in XCiT/CaiT).

In [16]:
class ViTMultitask(nn.Module):
    def __init__(self,
                 image_size  = IMAGE_SIZE,
                 patch_size  = PATCH_SIZE,
                 in_channels = IN_CHANNELS,
                 num_classes = NUM_CLASSES,
                 dim         = DIM,
                 depth       = DEPTH,
                 num_heads   = NUM_HEADS,
                 mlp_ratio   = MLP_RATIO,
                 dropout     = DROPOUT,
                 drop_path   = DROP_PATH):
        super().__init__()
        self.patch_size = patch_size
        self.H = self.W = image_size // patch_size
        num_patches     = self.H * self.W

        # Backbone (key names match ViTEncoder in Phase 1)
        self.patch_embed = nn.Conv2d(in_channels, dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.emb_norm    = nn.LayerNorm(dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # CLS token prepended for sequence classification
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Learned position for the CLS token slot
        self.cls_pos     = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.trunc_normal_(self.cls_pos, std=0.02)

        dpr = [x.item() for x in torch.linspace(0, drop_path, depth)]
        self.blocks = nn.ModuleList([
            ViTBlock(dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)

        # Three parallel heads (randomly initialised)
        self.head_cls  = nn.Linear(dim, num_classes)
        self.head_mass = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(dim//2, 1))
        self.head_pt   = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(dim//2, 1))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _backbone_params(self):
        head_ids = {id(p) for h in [self.head_cls,self.head_mass,self.head_pt]
                    for p in h.parameters()}
        return [p for p in self.parameters() if id(p) not in head_ids]

    def head_params(self):
        return [p for h in [self.head_cls,self.head_mass,self.head_pt]
                for p in h.parameters()]

    def freeze_backbone(self):
        for p in self._backbone_params(): p.requires_grad_(False)
        self.cls_token.requires_grad_(False)
        print(f'Backbone frozen ({sum(p.numel() for p in self._backbone_params()):,} params).')

    def unfreeze_backbone(self):
        for p in self._backbone_params(): p.requires_grad_(True)
        self.cls_token.requires_grad_(True)
        print('Backbone unfrozen.')

    def forward(self, x):
        B  = x.shape[0]
        H, W = self.H, self.W
        ps = self.patch_size
        pad_h = (ps - x.shape[2] % ps) % ps
        pad_w = (ps - x.shape[3] % ps) % ps
        if pad_h or pad_w: x = F.pad(x, (0, pad_w, 0, pad_h))

        # Patch tokens with positional encoding
        x = self.patch_embed(x).flatten(2).transpose(1, 2).contiguous()
        x = self.emb_norm(x + self.pos_embed)  # (B, N, dim)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1) + self.cls_pos
        x   = torch.cat([cls, x], dim=1)        # (B, N+1, dim)

        # ViT blocks over full sequence (CLS + patches)
        for blk in self.blocks:
            x = blk(x)
        x    = self.norm(x)
        feat = x[:, 0]  # CLS token output

        return self.head_cls(feat), self.head_mass(feat).squeeze(-1), self.head_pt(feat).squeeze(-1)


_m = ViTMultitask().to(DEVICE)
_x = torch.randn(2, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
_lg, _mp, _pp = _m(_x)
n_tot = sum(p.numel() for p in _m.parameters())
n_h   = sum(p.numel() for p in _m.head_params())
print(f'Forward OK  logits={_lg.shape}  mass={_mp.shape}  pt={_pp.shape}')
print(f'Params: total={n_tot:,}  backbone={n_tot-n_h:,}  heads={n_h:,}')
del _m, _x


Forward OK  logits=torch.Size([2, 2])  mass=torch.Size([2])  pt=torch.Size([2])
Params: total=3,756,292  backbone=3,718,656  heads=37,636


## 6 . Dataset & Loaders

In [17]:
class JetDatasetMT(Dataset):
    def __init__(self, X, y, mass, pt):
        self.X    = torch.from_numpy(X).float().permute(0, 3, 1, 2)
        self.y    = torch.from_numpy(y).long()
        self.mass = torch.from_numpy(mass).float()
        self.pt   = torch.from_numpy(pt).float()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i], self.mass[i], self.pt[i]

def make_loaders_mt(X_tr,y_tr,m_tr,p_tr,X_te,y_te,m_te,p_te):
    tr = DataLoader(JetDatasetMT(X_tr,y_tr,m_tr,p_tr),
                    BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    te = DataLoader(JetDatasetMT(X_te,y_te,m_te,p_te),
                    BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return tr, te


## 7 . Multitask Loss

In [18]:
def multitask_loss(logits, mp, pp, y, m_true, p_true, cls_crit):
    l_cls  = cls_crit(logits, y)
    l_mass = F.mse_loss(mp, m_true)
    l_pt   = F.mse_loss(pp, p_true)
    return W_CLS*l_cls + W_MASS*l_mass + W_PT*l_pt, l_cls, l_mass, l_pt


## 8 . Training & Evaluation Loops

In [19]:
def train_epoch_mt(model, loader, cls_crit, opt, scaler):
    model.train()
    tot = {'loss':0,'cls':0,'mass':0,'pt':0,'correct':0,'n':0}
    nb  = len(loader)
    for i,(X,y,mass,pt) in enumerate(loader):
        X,y,mass,pt = X.to(DEVICE),y.to(DEVICE),mass.to(DEVICE),pt.to(DEVICE)
        opt.zero_grad()
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type=='cuda')):
            lg,mp,pp = model(X)
            loss,lc,lm,lp = multitask_loss(lg,mp,pp,y,mass,pt,cls_crit)
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        scaler.step(opt); scaler.update()
        bs = len(y)
        tot['loss']+=loss.item()*bs; tot['cls']+=lc.item()*bs
        tot['mass']+=lm.item()*bs;   tot['pt']+=lp.item()*bs
        tot['correct']+=(lg.argmax(1)==y).sum().item(); tot['n']+=bs
        bar='X'*int(30*(i+1)/nb)+'.'*(30-int(30*(i+1)/nb))
        print(f'\r  [{bar}] {i+1}/{nb}  loss={tot["loss"]/tot["n"]:.4f}',end='',flush=True)
    print()
    n=tot['n']
    return tot['loss']/n,tot['cls']/n,tot['mass']/n,tot['pt']/n,tot['correct']/n


@torch.no_grad()
def eval_epoch_mt(model, loader, cls_crit, norm_stats):
    model.eval()
    mm,ms,pm,ps_std = norm_stats
    tot = {'loss':0,'cls':0,'mass':0,'pt':0,'correct':0,'n':0}
    ps_all,ls_all,mp_all,mt_all,pp_all,pt_all=[],[],[],[],[],[]
    for X,y,mass,pt in loader:
        X,y,mass,pt = X.to(DEVICE),y.to(DEVICE),mass.to(DEVICE),pt.to(DEVICE)
        lg,mp,pp = model(X)
        loss,lc,lm,lp = multitask_loss(lg,mp,pp,y,mass,pt,cls_crit)
        bs=len(y)
        tot['loss']+=loss.item()*bs; tot['cls']+=lc.item()*bs
        tot['mass']+=lm.item()*bs;   tot['pt']+=lp.item()*bs
        tot['correct']+=(lg.argmax(1)==y).sum().item(); tot['n']+=bs
        ps_all.append(lg.softmax(-1).cpu().numpy())
        ls_all.append(y.cpu().numpy())
        mp_all.append(mp.cpu().numpy()*ms+mm)
        mt_all.append(mass.cpu().numpy()*ms+mm)
        pp_all.append(pp.cpu().numpy()*ps_std+pm)
        pt_all.append(pt.cpu().numpy()*ps_std+pm)
    ps_all=np.concatenate(ps_all); ls_all=np.concatenate(ls_all)
    mp_g=np.concatenate(mp_all);   mt_g=np.concatenate(mt_all)
    pp_g=np.concatenate(pp_all);   pt_g=np.concatenate(pt_all)
    try:    auc=roc_auc_score(ls_all,ps_all[:,1])
    except: auc=0.5
    n=tot['n']
    return (tot['loss']/n,tot['cls']/n,tot['mass']/n,tot['pt']/n,
            tot['correct']/n,auc,
            np.abs(mp_g-mt_g).mean(),np.abs(pp_g-pt_g).mean(),
            ps_all,ls_all,mp_g,mt_g,pp_g,pt_g)


## 9 . Load Data

In [20]:
X_train    = np.load(f'{MT_DIR}/X_train_mt.npy')
X_test     = np.load(f'{MT_DIR}/X_test_mt.npy')
y_train    = np.load(f'{MT_DIR}/y_train_mt.npy')
y_test     = np.load(f'{MT_DIR}/y_test_mt.npy')
mass_train = np.load(f'{MT_DIR}/mass_train_norm.npy')
mass_test  = np.load(f'{MT_DIR}/mass_test_norm.npy')
pt_train   = np.load(f'{MT_DIR}/pt_train_norm.npy')
pt_test    = np.load(f'{MT_DIR}/pt_test_norm.npy')
norm_stats = np.load(f'{MT_DIR}/norm_stats.npy')
mass_mean, mass_std, pt_mean, pt_std = norm_stats

print(f'X_train    : {X_train.shape}')
print(f'y_train    : {y_train.shape}  classes={list(np.unique(y_train))}')
print(f'mass stats : mean={mass_mean:.2f} GeV  std={mass_std:.2f} GeV')
print(f'pT   stats : mean={pt_mean:.2f} GeV    std={pt_std:.2f} GeV')

pos_w    = float((y_train==0).sum()/(y_train==1).sum())
cls_crit = nn.CrossEntropyLoss(weight=torch.tensor([1.0, pos_w]).to(DEVICE))

train_loader, test_loader = make_loaders_mt(
    X_train, y_train, mass_train, pt_train,
    X_test,  y_test,  mass_test,  pt_test)
print(f'Train batches={len(train_loader)}  Test batches={len(test_loader)}')


X_train    : (8000, 125, 125, 8)
y_train    : (8000,)  classes=[np.int64(0), np.int64(1)]
mass stats : mean=142.78 GeV  std=51.51 GeV
pT   stats : mean=521.44 GeV    std=109.67 GeV
Train batches=500  Test batches=125


## 10 . Load ViT MAE Weights
The ViT encoder `state_dict` maps directly onto `ViTMultitask` backbone keys.
Only `head_cls`, `head_mass`, `head_pt` will appear in `missing_keys` -- expected.
`mask_token` (MAEDecoder) is filtered out if present.

In [21]:
model_ft = ViTMultitask().to(DEVICE)

state     = torch.load(VIT_WEIGHTS, map_location=DEVICE)
model_keys = set(model_ft.state_dict().keys())
state      = {k: v for k, v in state.items() if k in model_keys}

missing, unexpected = model_ft.load_state_dict(state, strict=False)
print('ViT MAE weights loaded.')
print(f'  Missing keys  (should be heads only): {[k for k in missing if "head" in k]}')
print(f'  Unexpected keys (should be empty)   : {unexpected}')

n_total = sum(p.numel() for p in model_ft.parameters())
n_heads = sum(p.numel() for p in model_ft.head_params())
print(f'\nTotal params : {n_total:,}  |  Backbone : {n_total-n_heads:,}  |  Heads : {n_heads:,}')
scaler_ft = torch.amp.GradScaler(enabled=(DEVICE.type=='cuda'))


ViT MAE weights loaded.
  Missing keys  (should be heads only): ['head_cls.weight', 'head_cls.bias', 'head_mass.0.weight', 'head_mass.0.bias', 'head_mass.3.weight', 'head_mass.3.bias', 'head_pt.0.weight', 'head_pt.0.bias', 'head_pt.3.weight', 'head_pt.3.bias']
  Unexpected keys (should be empty)   : []

Total params : 3,756,292  |  Backbone : 3,718,656  |  Heads : 37,636


## 11 . Step 1 -- Heads-only Warm-up (Backbone Frozen)

In [22]:
model_ft.freeze_backbone()

opt1   = optim.AdamW(model_ft.head_params(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
def _lr(ep, warmup, total):
    if ep < warmup: return (ep+1)/warmup
    return 0.5*(1+math.cos(math.pi*(ep-warmup)/max(1,total-warmup)))
sched1 = optim.lr_scheduler.LambdaLR(opt1, lambda ep: _lr(ep,WARMUP_EPOCHS,EPOCHS_FROZEN))

hist_ft = {k:[] for k in ['loss','cls','mass_mse','pt_mse','acc','auc','mass_mae','pt_mae']}
best_auc_ft, best_ep_ft = 0.0, 0

print(f'{'Ep':>4} {'Loss':>8} {'Cls':>7} {'MassMSE':>9} {'PtMSE':>8} {'Acc':>7} {'AUC':>7} {'MassMAE':>9} {'PtMAE':>8}')
print('='*75)

for ep in range(1, EPOCHS_FROZEN+1):
    t0 = time.time()
    print(f'Epoch {ep}/{EPOCHS_FROZEN} [heads only]')
    tr = train_epoch_mt(model_ft, train_loader, cls_crit, opt1, scaler_ft)
    va = eval_epoch_mt(model_ft, test_loader, cls_crit, norm_stats)
    sched1.step()
    loss,cls,mmse,pmse,acc = tr
    _,_,_,_,_,auc,mmae,pmae,*_ = va
    for k,v in zip(hist_ft,[loss,cls,mmse,pmse,acc,auc,mmae,pmae]): hist_ft[k].append(v)
    flag = ' *** BEST' if auc > best_auc_ft else ''
    print(f'{ep:>4} {loss:>8.4f} {cls:>7.4f} {mmse:>9.4f} {pmse:>8.4f} {acc:>7.4f} {auc:>7.4f} {mmae:>7.2f}GeV {pmae:>6.2f}GeV {time.time()-t0:.1f}s{flag}')
    if auc > best_auc_ft:
        best_auc_ft, best_ep_ft = auc, ep
        torch.save(model_ft.state_dict(), 'best_vit_mt_ft.pth')

print(f'\nStep 1 done. Best AUC={best_auc_ft:.4f} at epoch {best_ep_ft}')


Backbone frozen (3,718,656 params).
  Ep     Loss     Cls   MassMSE    PtMSE     Acc     AUC   MassMAE    PtMAE
Epoch 1/3 [heads only]
  [XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX] 500/500  loss=3.2839
   1   3.2839  0.6600    0.9631   1.0008  0.6078  0.7071   40.12GeV  76.81GeV 153.1s *** BEST
Epoch 2/3 [heads only]


/Users/abhirajraje/Desktop/E2E_Linear_ViT/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX] 500/500  loss=3.2173
   2   3.2173  0.6371    0.9422   1.0009  0.6331  0.7153   39.67GeV  74.74GeV 157.1s *** BEST
Epoch 3/3 [heads only]


/Users/abhirajraje/Desktop/E2E_Linear_ViT/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX] 500/500  loss=3.2033
   3   3.2033  0.6343    0.9390   0.9957  0.6366  0.7221   40.08GeV  74.58GeV 151.4s *** BEST

Step 1 done. Best AUC=0.7221 at epoch 3


## 12 . Step 2 -- Full Fine-tuning (Backbone Unfrozen at Low LR)
Backbone updated at `LR_BACKBONE_FT = 3e-5`, heads continue at `LR_HEAD = 1e-3`.
Identical protocol to `xcit-multitask.ipynb`.

In [23]:
model_ft.unfreeze_backbone()

no_decay = {'bias','LayerNorm.weight','LayerNorm.bias','norm.weight','norm.bias','pos_embed','cls_token','cls_pos'}
head_ids = {id(p) for p in model_ft.head_params()}

opt2 = optim.AdamW([
    {'params':[p for n,p in model_ft.named_parameters()
               if id(p) not in head_ids and not any(nd in n for nd in no_decay)],
     'lr':LR_BACKBONE_FT, 'weight_decay':WEIGHT_DECAY},
    {'params':[p for n,p in model_ft.named_parameters()
               if id(p) not in head_ids and     any(nd in n for nd in no_decay)],
     'lr':LR_BACKBONE_FT, 'weight_decay':0.0},
    {'params':model_ft.head_params(), 'lr':LR_HEAD, 'weight_decay':WEIGHT_DECAY},
], betas=(0.9, 0.999))
sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_FINETUNE)

print(f'{'Ep':>4} {'Loss':>8} {'Cls':>7} {'MassMSE':>9} {'PtMSE':>8} {'Acc':>7} {'AUC':>7} {'MassMAE':>9} {'PtMAE':>8}')
print('='*75)

for ep in range(1, EPOCHS_FINETUNE+1):
    t0 = time.time()
    print(f'Epoch {ep}/{EPOCHS_FINETUNE} [end-to-end]')
    tr = train_epoch_mt(model_ft, train_loader, cls_crit, opt2, scaler_ft)
    va = eval_epoch_mt(model_ft, test_loader, cls_crit, norm_stats)
    sched2.step()
    loss,cls,mmse,pmse,acc = tr
    _,_,_,_,_,auc,mmae,pmae,*_ = va
    for k,v in zip(hist_ft,[loss,cls,mmse,pmse,acc,auc,mmae,pmae]): hist_ft[k].append(v)
    flag = ' *** BEST' if auc > best_auc_ft else ''
    print(f'{ep:>4} {loss:>8.4f} {cls:>7.4f} {mmse:>9.4f} {pmse:>8.4f} {acc:>7.4f} {auc:>7.4f} {mmae:>7.2f}GeV {pmae:>6.2f}GeV {time.time()-t0:.1f}s{flag}')
    if auc > best_auc_ft:
        best_auc_ft, best_ep_ft = auc, ep
        torch.save(model_ft.state_dict(), 'best_vit_mt_ft.pth')

print(f'\nFine-tune done. Best AUC={best_auc_ft:.4f} at epoch {best_ep_ft}')


Backbone unfrozen.
  Ep     Loss     Cls   MassMSE    PtMSE     Acc     AUC   MassMAE    PtMAE
Epoch 1/1 [end-to-end]


/Users/abhirajraje/Desktop/E2E_Linear_ViT/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [XXXXXXXXXXXXXXXXXXXXX.........] 356/500  loss=2.9488

KeyboardInterrupt: 

## 13 . Scratch Baseline

In [ ]:
model_sc = ViTMultitask().to(DEVICE)
n = sum(p.numel() for p in model_sc.parameters())
print(f'Scratch params: {n:,}')

no_decay = {'bias','LayerNorm.weight','LayerNorm.bias','norm.weight','norm.bias','pos_embed','cls_token','cls_pos'}
opt_sc = optim.AdamW([
    {'params':[p for nm,p in model_sc.named_parameters() if not any(nd in nm for nd in no_decay)],
     'weight_decay':WEIGHT_DECAY, 'lr':LR_SCRATCH},
    {'params':[p for nm,p in model_sc.named_parameters() if     any(nd in nm for nd in no_decay)],
     'weight_decay':0.0, 'lr':LR_SCRATCH},
])
sched_sc  = optim.lr_scheduler.LambdaLR(opt_sc, lambda ep: _lr(ep,WARMUP_EPOCHS,EPOCHS_SCRATCH))
scaler_sc = torch.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

hist_sc = {k:[] for k in ['loss','cls','mass_mse','pt_mse','acc','auc','mass_mae','pt_mae']}
best_auc_sc, best_ep_sc = 0.0, 0

print(f'{'Ep':>4} {'Loss':>8} {'Cls':>7} {'MassMSE':>9} {'PtMSE':>8} {'Acc':>7} {'AUC':>7} {'MassMAE':>9} {'PtMAE':>8}')
print('='*75)

for ep in range(1, EPOCHS_SCRATCH+1):
    t0 = time.time()
    print(f'Epoch {ep}/{EPOCHS_SCRATCH} [scratch]')
    tr = train_epoch_mt(model_sc, train_loader, cls_crit, opt_sc, scaler_sc)
    va = eval_epoch_mt(model_sc, test_loader, cls_crit, norm_stats)
    sched_sc.step()
    loss,cls,mmse,pmse,acc = tr
    _,_,_,_,_,auc,mmae,pmae,*_ = va
    for k,v in zip(hist_sc,[loss,cls,mmse,pmse,acc,auc,mmae,pmae]): hist_sc[k].append(v)
    flag = ' *** BEST' if auc > best_auc_sc else ''
    print(f'{ep:>4} {loss:>8.4f} {cls:>7.4f} {mmse:>9.4f} {pmse:>8.4f} {acc:>7.4f} {auc:>7.4f} {mmae:>7.2f}GeV {pmae:>6.2f}GeV {time.time()-t0:.1f}s{flag}')
    if auc > best_auc_sc:
        best_auc_sc, best_ep_sc = auc, ep
        torch.save(model_sc.state_dict(), 'best_vit_mt_sc.pth')

print(f'\nScratch done. Best AUC={best_auc_sc:.4f} at epoch {best_ep_sc}')


## 14 . Final Evaluation & Comparison

In [ ]:
def final_eval(model, path, name):
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    out = eval_epoch_mt(model, test_loader, cls_crit, norm_stats)
    _,_,_,_,acc,auc,mmae,pmae,ps,ls,mp_g,mt_g,pp_g,pt_g = out
    preds = np.argmax(ps, axis=1)
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Classification  acc={acc:.4f}  AUC={auc:.4f}')
    print(f'  Mass MAE={mmae:.3f} GeV   r={np.corrcoef(mt_g,mp_g)[0,1]:.3f}')
    print(f'  pT   MAE={pmae:.3f} GeV   r={np.corrcoef(pt_g,pp_g)[0,1]:.3f}')
    print(classification_report(ls, preds, target_names=['class 0','class 1']))
    return mp_g, mt_g, pp_g, pt_g


mp_ft,mt_ft,pp_ft,pt_ft = final_eval(model_ft,'best_vit_mt_ft.pth','ViT FINETUNE (MAE pretrained)')
mp_sc,mt_sc,pp_sc,pt_sc = final_eval(model_sc,'best_vit_mt_sc.pth','ViT SCRATCH')


## 15 . Training Curves

In [ ]:
fig,axes = plt.subplots(2,4,figsize=(20,8))
metrics = ['loss','cls','mass_mse','pt_mse','acc','auc','mass_mae','pt_mae']
titles  = ['Total Loss','Cls Loss','Mass MSE','pT MSE','Accuracy','AUC','Mass MAE (GeV)','pT MAE (GeV)']
for ax,m,title in zip(axes.flat,metrics,titles):
    ax.plot(hist_ft[m], label='ViT finetune')
    ax.plot(hist_sc[m], label='ViT scratch', linestyle='--')
    ax.set_title(title); ax.legend(); ax.grid(True)
plt.suptitle('ViT Baseline: Finetune vs Scratch -- Multitask', fontsize=14)
plt.tight_layout()
plt.savefig('vit_mt_curves.png', dpi=150); plt.show()


## 16 . Regression Scatter Plots

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(12,10))
for ax,(pred,true,label,mname) in zip(axes.flat,[
    (mp_ft,mt_ft,'Mass (GeV)','ViT Finetune'),
    (pp_ft,pt_ft,'pT (GeV)',  'ViT Finetune'),
    (mp_sc,mt_sc,'Mass (GeV)','ViT Scratch'),
    (pp_sc,pt_sc,'pT (GeV)',  'ViT Scratch')]):
    ax.scatter(true,pred,alpha=0.3,s=5)
    lims=[min(true.min(),pred.min()),max(true.max(),pred.max())]
    ax.plot(lims,lims,'r--',linewidth=1.5,label='ideal')
    ax.set_xlabel(f'True {label}'); ax.set_ylabel(f'Predicted {label}')
    mae=np.abs(pred-true).mean(); r=np.corrcoef(true,pred)[0,1]
    ax.set_title(f'{mname} -- {label}\nMAE={mae:.2f}  r={r:.3f}')
    ax.legend(); ax.grid(True)
plt.suptitle('ViT Baseline: Predicted vs True -- Mass & pT', fontsize=14)
plt.tight_layout()
plt.savefig('vit_mt_regression.png', dpi=150); plt.show()


## 17 . Attention Map Visualisation
For the ViT baseline we visualise the **CLS token attention weights** from the last
transformer block -- the standard ViT interpretability approach.
This is the direct analogue of XCiT's Q/K norm saliency.

In [ ]:
@torch.no_grad()
def vit_attention_map(model, img_tensor):
    model.eval()
    x  = img_tensor.to(DEVICE)
    ps = model.patch_size
    H, W = model.H, model.W
    pad_h = (ps - x.shape[2]%ps)%ps
    pad_w = (ps - x.shape[3]%ps)%ps
    if pad_h or pad_w: x = F.pad(x,(0,pad_w,0,pad_h))

    B = x.shape[0]
    # Build sequence with CLS
    x_emb = model.patch_embed(x).flatten(2).transpose(1,2).contiguous()
    x_emb = model.emb_norm(x_emb + model.pos_embed)
    cls   = model.cls_token.expand(B,-1,-1) + model.cls_pos
    x_seq = torch.cat([cls, x_emb], dim=1)   # (B, N+1, dim)

    # Pass through all but last block
    for blk in model.blocks[:-1]:
        x_seq = blk(x_seq)

    # Extract raw attention from last block
    last  = model.blocks[-1]
    x_n   = last.norm1(x_seq)
    B2, N1, C = x_n.shape
    h = last.attn.num_heads
    qkv = last.attn.qkv(x_n).reshape(B2,N1,3,h,C//h).permute(2,0,3,1,4)
    q, k = qkv[0], qkv[1]                          # (B, h, N+1, d/h)
    attn = (q @ k.transpose(-2,-1)) * last.attn.scale
    attn = attn.softmax(dim=-1)                     # (B, h, N+1, N+1)

    # CLS-to-patch attention: row 0 of the attention matrix, columns 1:
    cls_attn = attn[0, :, 0, 1:]                    # (h, N)
    cls_attn = cls_attn.mean(0).cpu().numpy()        # (N,)
    return cls_attn.reshape(H, W)


model_ft.load_state_dict(torch.load('best_vit_mt_ft.pth', map_location=DEVICE))
sample_t = torch.from_numpy(X_test[0:1]).float().permute(0,3,1,2)
attn_map = vit_attention_map(model_ft, sample_t)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(X_test[0,:,:,0], cmap='viridis')
ax[0].set_title(f'Jet ch0 (label={y_test[0]})'); ax[0].axis('off')
im = ax[1].imshow(attn_map, cmap='hot')
ax[1].set_title('CLS attention (last block, avg over heads)')
plt.colorbar(im, ax=ax[1])
plt.suptitle('ViT Multitask: CLS Token Attention Map', fontsize=13)
plt.tight_layout()
plt.savefig('vit_mt_attention.png', dpi=150); plt.show()
